# Spam Detector - EDA and Modeling

This notebook explores the UCI SMS Spam Collection dataset and compares two baseline models: Multinomial Naive Bayes and Linear SVM. It also saves the best model to `models/best_model.joblib`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Allow importing from src/ without installing a package
sys.path.append(str(Path("src").resolve()))

data_path = Path("data/processed/sms.csv")
if not data_path.exists():
    print("Missing data/processed/sms.csv. Run: python scripts/download_prepare.py --download --prepare")

df = pd.read_csv(data_path) if data_path.exists() else None
df.head() if df is not None else None

In [ ]:
if df is not None:
    import matplotlib.pyplot as plt
    import seaborn as sns

    sns.countplot(data=df, x="label")
    plt.title("Class distribution")
    plt.show()

In [ ]:
from collections import Counter
from spamdetector.pipeline import clean_text

def top_words(texts, n=20):
    counter = Counter()
    for text in texts:
        counter.update(clean_text(text).split())
    return counter.most_common(n)

if df is not None:
    spam_words = top_words(df[df["label"] == "spam"]["text"])
    ham_words = top_words(df[df["label"] == "ham"]["text"])

    print("Top spam words:", spam_words)
    print("Top ham words:", ham_words)

In [ ]:
from spamdetector.train import train_and_select

if df is not None:
    best_model, results, models = train_and_select(
        data_path=str(data_path),
        test_size=0.2,
        random_state=42,
        use_stemming=False,
        use_stopwords=True,
        use_nltk_stopwords=True,
        calibrate_svm=True,
    )

    for name, metrics in results.items():
        print(f"Model: {name}")
        print(f"  Accuracy: {metrics['accuracy']:.4f}")
        print(f"  Spam precision: {metrics['spam']['precision']:.4f}")
        print(f"  Spam recall: {metrics['spam']['recall']:.4f}")
        print(f"  Spam F1: {metrics['spam']['f1']:.4f}")
        print(f"  Confusion matrix: {metrics['confusion_matrix']}")
        print()

    print(f"Best model: {best_model}")

In [ ]:
from spamdetector.train import save_model

if df is not None:
    model_output = Path("models/best_model.joblib")
    save_model(models[best_model], str(model_output))
    print(f"Saved model to {model_output}")